In [1]:
"""
Descarga de clasificación del suelo para análisis de construcción
planta de biometano — Provincia de Huesca

Fuentes:
  1. ICEAragón WFS (SIUa) — clasificación urbanística

Output: clasificacion_suelo_huesca.gpkg

NOTA sobre clasificacion_idearagon:
  La capa SIUa:clasificacion_idearagon NO está disponible vía GetFeature JSON
  en el endpoint SIUa_WMS. Se prueban variantes de nombre y parámetros.
  Si todo falla, descarga manual en: https://idearagon.aragon.es/SIUa/descargas
"""

import geopandas as gpd
import requests
import io
from pathlib import Path
from shapely.ops import unary_union

# ─────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────
DELIM_DIR      = Path("../../data/raw/delimitations")
OUTPUT_DIR     = Path("../../data/raw/06_clasificacion_suelo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEOJSON_HUESCA = DELIM_DIR / "Huesca_Delimitacion.geojson"
OUTPUT_GPKG    = OUTPUT_DIR / "clasificacion_suelo_huesca.gpkg"

CRS_UTM    = "EPSG:25830"
BBOX_WGS84 = (-0.934168, 41.347806, 0.771832, 42.921806)  # minx,miny,maxx,maxy
HEADERS    = {"User-Agent": "biometano-huesca-research/1.0 (academic)"}
WFS_SIUA = "https://icearagon.aragon.es/geoserver/wfs"


def cargar_huesca():
    huesca = gpd.read_file(GEOJSON_HUESCA).to_crs(CRS_UTM)
    return unary_union(huesca.geometry)


# ─────────────────────────────────────────────────────────
# 1. ICEAragón SIUa — CLASIFICACIÓN URBANÍSTICA
# ─────────────────────────────────────────────────────────
def descargar_clasificacion_urbanistica():
    print("Descargando clasificación urbanística (ICEAragón SIUa)...")

    WFS_URL = "https://icearagon.aragon.es/geoserver/wfs"

    # BBOX de Huesca en EPSG:25830 (UTM zona 30N)
    # minx, miny, maxx, maxy
    BBOX_UTM = (585000, 4580000, 790000, 4760000)

    params = {
        "SERVICE":      "WFS",
        "VERSION":      "2.0.0",
        "REQUEST":      "GetFeature",
        "TYPENAMES":    "SIUa:clasificacion_idearagon",
        "OUTPUTFORMAT": "application/json",
        "SRSNAME":      "EPSG:25830",
        "BBOX":         f"{BBOX_UTM[0]},{BBOX_UTM[1]},{BBOX_UTM[2]},{BBOX_UTM[3]},EPSG:25830",
    }
    try:
        r = requests.get(WFS_URL, params=params, timeout=300, headers=HEADERS)
        r.raise_for_status()

        if b"ExceptionReport" in r.content[:500]:
            print(f"  ExceptionReport:\n{r.text}")
            return None

        gdf = gpd.read_file(io.BytesIO(r.content))

        if gdf is None or len(gdf) == 0:
            print("  Respuesta vacía")
            return None

        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:25830")
        gdf = gdf.to_crs(CRS_UTM)
        print(f"  Total: {len(gdf)} polígonos")
        col_cls = next((c for c in gdf.columns
                        if any(k in c.lower() for k in
                               ["clasif", "tipo", "clase", "uso"])), None)
        if col_cls:
            print(f"  Valores '{col_cls}': {gdf[col_cls].value_counts().to_dict()}")
        return gdf

    except requests.HTTPError as e:
        print(f"  Error HTTP {e.response.status_code}:\n{e.response.text[:800]}")
    except Exception as e:
        print(f"  Error: {e}")

    print("  Sin datos. Descarga manual:")
    print("  → https://idearagon.aragon.es/SIUa/descargas  (seleccionar Huesca, formato GPKG)")
    return None


# ─────────────────────────────────────────────────────────
# GUARDAR
# ─────────────────────────────────────────────────────────
def guardar(gdf_urbanistico):
    print()
    capas = {
        "clasificacion_urbanistica": gdf_urbanistico,
    }
    for nombre, gdf in capas.items():
        if gdf is not None and len(gdf) > 0:
            gdf.to_file(OUTPUT_GPKG, driver="GPKG", layer=nombre)
            print(f"  '{nombre}': {len(gdf)} entidades → {OUTPUT_GPKG}")
        else:
            print(f"  '{nombre}': sin datos, no guardada")


# ─────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("  CLASIFICACIÓN SUELO HUESCA — DESCARGA")
    print("=" * 60 + "\n")

    gdf_urbanistico = descargar_clasificacion_urbanistica()

    guardar(gdf_urbanistico)

    print("\nHecho. Capas en:", OUTPUT_GPKG)


  CLASIFICACIÓN SUELO HUESCA — DESCARGA

Descargando clasificación urbanística (ICEAragón SIUa)...
  Total: 33527 polígonos
  Valores 'clasesiose': {'NULL': 393, 'SU-C': 180, 'SNU-G': 100, 'SU': 88, 'SNU-C': 1, 'SNU-E': 1}

  'clasificacion_urbanistica': 33527 entidades → ..\..\data\raw\06_clasificacion_suelo\clasificacion_suelo_huesca.gpkg

Hecho. Capas en: ..\..\data\raw\06_clasificacion_suelo\clasificacion_suelo_huesca.gpkg


In [2]:
import numpy as np
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path
from shapely.ops import unary_union

RAW_DIR   = Path("../../data/raw/06_clasificacion_suelo")
DELIM_DIR = Path("../../data/raw/delimitations")
MAP_DIR   = Path("../../docs/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf = gdf[gdf["cod_ine"].astype(str).str.startswith("22")].copy()
gdf = gdf.to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].buffer(0)  # repara micro-autointersecciones antes de operar
gdf["geometry"] = gdf["geometry"].simplify(0.0003, preserve_topology=True)
gdf_clasificado = gdf[gdf["clase"].notna()].copy()

CLASE_STYLE = {
    "SUC":   {"color": [52,  152, 219, 255]},   # azul vivo
    "SUNC":  {"color": [41,  128, 185, 255]},   # azul oscuro
    "SDUD":  {"color": [243, 156,  18, 255]},   # naranja vivo
    "SDUNC": {"color": [230, 126,  34, 255]},   # naranja oscuro
    "SENU":  {"color": [ 39, 174,  96, 140]},   # verde — más transparente
    "SENE":  {"color": [ 26, 188, 156, 140]},   # verde agua — más transparente
    "SEUND": {"color": [155,  89, 182, 255]},   # morado vivo
}
DEFAULT = {"color": [150, 150, 150, 150]}

# ─────────────────────────────────────────────────────────
# 1. Resolver solapes entre parcelas (misma técnica que en el
#    mapa de Natura 2000: recortar por prioridad), PERO a nivel
#    de parcela individual, para no perder los atributos del
#    tooltip (clase_s, municipio, uso_glob) al dissolver.
#
#    Prioridad: urbano consolidado/no consolidado primero
#    (son los datos más específicos y fiables), luego
#    urbanizable, luego no urbanizable especial, y por último
#    el genérico SENU (la clase "por defecto", la que más
#    parcelas duplicadas suele arrastrar).
# ─────────────────────────────────────────────────────────
PRIORIDAD = ["SUC", "SUNC", "SDUD", "SDUNC", "SEUND", "SENE", "SENU"]

gdf_clasificado["clase"] = gdf_clasificado["clase"].astype(str).str.strip()
resolved_parts = []
cubierto = None

for clase in PRIORIDAD:
    tier = gdf_clasificado[gdf_clasificado["clase"] == clase].copy()
    if tier.empty:
        continue
    if cubierto is not None:
        tier["geometry"] = tier["geometry"].difference(cubierto)
        tier = tier[~tier["geometry"].is_empty].copy()
    if tier.empty:
        continue
    resolved_parts.append(tier)
    nuevo = unary_union(tier.geometry)
    cubierto = nuevo if cubierto is None else cubierto.union(nuevo)

# Clases no contempladas en PRIORIDAD (por si aparece algún código nuevo)
resto = gdf_clasificado[~gdf_clasificado["clase"].isin(PRIORIDAD)].copy()
if not resto.empty and cubierto is not None:
    resto["geometry"] = resto["geometry"].difference(cubierto)
    resto = resto[~resto["geometry"].is_empty].copy()
if not resto.empty:
    resolved_parts.append(resto)

gdf_resuelto = pd.concat(resolved_parts, ignore_index=True) if resolved_parts else gdf_clasificado.iloc[0:0]
gdf_resuelto = gpd.GeoDataFrame(gdf_resuelto, geometry="geometry", crs="EPSG:4326")

gdf_resuelto["fill_color"] = gdf_resuelto["clase"].apply(
    lambda c: CLASE_STYLE.get(c, DEFAULT)["color"]
)

# ─────────────────────────────────────────────────────────
# 2. Zonas sin dato (manchas negras) — en vez de dejarlas sin
#    dibujar, se calcula el hueco real (límite provincial menos
#    todo lo clasificado) y se pinta como categoría explícita
#    "Sin datos SIUa", en vez de dejar que se vea como agujero
#    sobre el basemap oscuro.
# ─────────────────────────────────────────────────────────
provincia = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:4326")
provincia_geom = unary_union(provincia.geometry).buffer(0)

cubierto_total = unary_union(gdf["geometry"].buffer(0))  # todo lo que SÍ tiene geometría (clasificada o no)
hueco = provincia_geom.difference(cubierto_total)

SIN_DATOS_COLOR = [90, 90, 95, 120]

def geom_to_rows(geom, clase_label):
    if geom is None or geom.is_empty:
        return []
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms) if geom.geom_type == "MultiPolygon" else []
    rows = []
    for p in polys:
        if p.is_empty or len(list(p.exterior.coords)) < 3:
            continue
        rows.append({
            "geometry": p,
            "clase": clase_label,
            "clase_s": "Sin datos",
            "d_muni_ine": "—",
            "uso_glob": "—",
            "fill_color": SIN_DATOS_COLOR,
        })
    return rows

filas_hueco = geom_to_rows(hueco, "SIN_DATOS")
gdf_hueco = gpd.GeoDataFrame(filas_hueco, geometry="geometry", crs="EPSG:4326") if filas_hueco else None

# ─────────────────────────────────────────────────────────
# 3. Unir todo y convertir a coordenadas para pydeck
# ─────────────────────────────────────────────────────────
def geom_to_coords_multi(geom):
    """Devuelve TODOS los anillos exteriores de un (Multi)Polygon, no solo
    el más grande — evita perder fragmentos tras el recorte de solapes."""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        if len(list(geom.exterior.coords)) < 3:
            return None
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        rings = [list(p.exterior.coords) for p in geom.geoms
                 if not p.is_empty and len(list(p.exterior.coords)) >= 3]
        return rings if rings else None
    return None

capas_finales = [gdf_resuelto] + ([gdf_hueco] if gdf_hueco is not None else [])
gdf_final = pd.concat(capas_finales, ignore_index=True)
gdf_final = gpd.GeoDataFrame(gdf_final, geometry="geometry", crs="EPSG:4326")
gdf_final["rings"] = gdf_final["geometry"].apply(geom_to_coords_multi)
gdf_final = gdf_final[gdf_final["rings"].notna()].copy()

# Explode: una fila por anillo/parte, conservando los atributos originales
# (necesario para no perder fragmentos de MultiPolygon en pydeck)
filas_explode = []
for _, row in gdf_final.iterrows():
    for ring in row["rings"]:
        filas_explode.append({
            "coordinates": [ring],
            "clase":       row["clase"],
            "clase_s":     row["clase_s"],
            "d_muni_ine":  row["d_muni_ine"],
            "uso_glob":    row["uso_glob"],
            "fill_color":  row["fill_color"],
        })

df = pd.DataFrame(filas_explode)
df["clase"]      = df["clase"].fillna("—")
df["clase_s"]    = df["clase_s"].fillna("—")
df["d_muni_ine"] = df["d_muni_ine"].fillna("—")
df["uso_glob"]   = df["uso_glob"].fillna("—")

print(f"Polígonos clasificados: {len(gdf_resuelto):,}")
print(f"Polígonos 'sin datos':  {len(gdf_hueco) if gdf_hueco is not None else 0:,}")
print(f"Polígonos totales:      {len(df):,}")

layer = pdk.Layer(
    "PolygonLayer",
    data=df,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color=[60, 60, 60, 100],
    line_width_min_pixels=0.3,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    wireframe=False,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <b>Clase:</b> {clase}<br/>
        <b>Subtipo:</b> {clase_s}<br/>
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Uso global:</b> {uso_glob}
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

LEGEND_HTML = """
<div style="position:fixed;bottom:30px;left:20px;background:rgba(0,0,0,0.75);
            color:#fff;padding:12px 16px;border-radius:8px;font-size:12px;
            font-family:sans-serif;z-index:9999;line-height:2">
  <b>Clasificación del suelo — Huesca</b><br/>
  <span style="color:#3498db">██</span> SUC   — Urbano Consolidado<br/>
  <span style="color:#2980b9">██</span> SUNC  — Urbano No Consolidado<br/>
  <span style="color:#f39c12">██</span> SDUD  — Urbanizable Delimitado<br/>
  <span style="color:#e67e22">██</span> SDUNC — Urbanizable No Delimitado<br/>
  <span style="color:#27ae60">██</span> SENU  — No Urbanizable Genérico<br/>
  <span style="color:#1abc9c">██</span> SENE  — No Urbanizable Especial<br/>
  <span style="color:#9b59b6">██</span> SEUND — No Urbanizable Especial ND<br/>
  <span style="color:#5a5a5f">██</span> Sin datos SIUa<br/>
</div>
"""

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

# ─────────────────────────────────────────────────────────
# Título, fuente y escala/norte (misma vestimenta que el resto
# de mapas del proyecto) — no se toca la leyenda ni la paleta
# ─────────────────────────────────────────────────────────
import math

INITIAL_ZOOM = view.zoom
INITIAL_LAT  = view.latitude

m_per_px = 156543.03392 * math.cos(math.radians(INITIAL_LAT)) / (2 ** INITIAL_ZOOM)
scale_km_options = [1, 2, 5, 10, 20, 25, 50]
target_px = 110
scale_km = min(scale_km_options, key=lambda k: abs((k * 1000 / m_per_px) - target_px))
scale_px = round((scale_km * 1000) / m_per_px)

overlay_extra_html = f"""
<style>
  .map-panel {{
    font-family: 'Inter','Segoe UI',sans-serif;
    background: rgba(18,20,24,0.82);
    backdrop-filter: blur(6px);
    color: #f2f2f2;
    border: 1px solid rgba(255,255,255,0.10);
    border-radius: 10px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.35);
  }}
</style>

<div class="map-panel" style="position:fixed;top:18px;left:18px;padding:14px 20px;
            max-width:380px;z-index:9999;">
  <div style="font-size:16px;font-weight:600;letter-spacing:0.2px;">
    Clasificación urbanística del suelo — Huesca
  </div>
  <div style="font-size:12px;color:rgba(255,255,255,0.6);margin-top:3px;">
    Suelo urbano, urbanizable y no urbanizable por parcela
  </div>
</div>

<div style="position:fixed;top:18px;right:18px;font-family:'Inter','Segoe UI',sans-serif;
            font-size:10.5px;color:rgba(255,255,255,0.45);z-index:9999;text-align:right;">
  Fuente: ICEAragón — SIUa (Sistema de Información Urbanística de Aragón)<br/>
  TFM — Análisis de idoneidad, planta de biometano
</div>

<div class="map-panel" style="position:fixed;bottom:26px;right:18px;padding:12px 18px;
            z-index:9999;">
  <div style="display:flex;align-items:center;gap:14px;">
    <svg width="30" height="30" viewBox="0 0 30 30">
      <line x1="15" y1="26" x2="15" y2="6" stroke="#f2f2f2" stroke-width="1.6"/>
      <polygon points="15,3 11,12 19,12" fill="#f2f2f2"/>
      <text x="15" y="0" font-size="9" fill="#f2f2f2" text-anchor="middle"
            font-family="Inter,sans-serif" font-weight="600">N</text>
    </svg>
    <div>
      <div style="width:{scale_px}px;height:0;border-top:2px solid #f2f2f2;position:relative;">
        <div style="position:absolute;left:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
        <div style="position:absolute;right:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
      </div>
      <div style="font-size:11px;color:rgba(255,255,255,0.75);margin-top:4px;">
        ≈ {scale_km} km <span style="color:rgba(255,255,255,0.45);">(zoom inicial)</span>
      </div>
    </div>
  </div>
</div>
"""

output_html = MAP_DIR / "mapa_clasificacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", LEGEND_HTML + overlay_extra_html + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Polígonos clasificados: 17,216
Polígonos 'sin datos':  26,474
Polígonos totales:      45,921
Abre: ..\..\docs\map\06_clasificacion_suelo\mapa_clasificacion_huesca.html


In [3]:
import numpy as np
import requests
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path
import unicodedata
import re

RAW_DIR = Path("../../data/raw/06_clasificacion_suelo")
MAP_DIR = Path("../../docs/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Descargar padrón INE ---
print("Descargando padrón INE...")
url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2875?nult=1"
r = requests.get(url, timeout=60)
data = r.json()

registros = []
for item in data:
    nombre = item.get("Nombre", "")
    valor  = item.get("Data", [{}])[0].get("Valor", None)
    registros.append({"nombre": nombre, "poblacion": valor})

raw_df = pd.DataFrame(registros)

# La fila de TOTAL PROVINCIA tiene el mismo texto "Huesca." que el MUNICIPIO de Huesca
# capital, pero la provincia aparece siempre como la primera fila "Huesca. ..." de toda
# la tabla. La identificamos por posición (índice mínimo), no por texto, para no
# confundirla con el municipio homónimo.
es_fila_huesca_texto = raw_df["nombre"].str.strip().str.match(r"^Huesca\.\s")
idx_provincia = raw_df[es_fila_huesca_texto].index.min()

padron = raw_df.drop(index=idx_provincia).copy()

# Solo filas "Total" (sexo combinado) + "Total habitantes" (todas las edades)
padron = padron[
    padron["nombre"].str.contains(r"\. Total\. Total habitantes", regex=True)
].copy()

def quitar_tildes(s):
    """Quita diacríticos para comparación robusta (Á->A, Í->I, É->E, etc.)"""
    return "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

def limpiar_nombre_ine(nombre_completo):
    """
    Convierte el nombre crudo del INE en un nombre normalizado y comparable
    con la capa geográfica, gestionando:
      - el artículo pospuesto con coma: "Fueva, La" -> "La Fueva"
      - el nombre bilingüe con barra: "Huesca/Uesca" -> "Huesca"
    """
    base = nombre_completo.split(".")[0].strip()
    base = base.split("/")[0].strip()

    m = re.match(r"^(.*),\s*(La|El|Los|Las)$", base, flags=re.IGNORECASE)
    if m:
        cuerpo, articulo = m.groups()
        base = f"{articulo} {cuerpo}"

    return base

padron["municipio_raw"] = padron["nombre"].apply(limpiar_nombre_ine)

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKC", str(s))
    s = quitar_tildes(s)             # Á->A, Í->I, etc. — clave para AÍSA/AISA, COSTEÁN/COSTEAN
    s = s.replace("-", " ")          # unifica guión y espacio
    s = " ".join(s.split())          # colapsa espacios múltiples/invisibles y hace strip
    return s.upper()

padron["municipio"] = padron["municipio_raw"].apply(normalizar_texto)
padron["poblacion"] = pd.to_numeric(padron["poblacion"], errors="coerce").fillna(0).astype(int)
padron = padron[["municipio", "poblacion"]]

dup = padron["municipio"].duplicated().sum()
if dup:
    print(f"⚠ {dup} municipios con más de una fila en el padrón tras filtrar — revisa el filtro")
padron = padron.drop_duplicates(subset="municipio", keep="first")

print(f"Municipios con población: {len(padron)}")

# --- 2. Buffer por población según normativa Aragón ---
def buffer_por_poblacion(pop):
    if pop < 3000: return 1000
    else:          return 2000

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf_raw = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf_raw = gdf_raw[gdf_raw["cod_ine"].astype(str).str.startswith("22")]
nucleos = gdf_raw[gdf_raw["clase"].isin(["SUC", "SUNC"])].copy()

nucleos["d_muni_ine"] = nucleos["d_muni_ine"].apply(normalizar_texto)  # ahora también quita tildes
nucleos["cod_ine"] = nucleos["cod_ine"].astype(str).str.strip()

nucleos_diss = nucleos.dissolve(by="cod_ine", aggfunc={"d_muni_ine": "first"}).reset_index()
nucleos_diss["municipio_key"] = nucleos_diss["d_muni_ine"].str.strip().str.upper()

dup_cod = nucleos_diss["cod_ine"].duplicated().sum()
if dup_cod:
    print(f"⚠ {dup_cod} cod_ine duplicados tras el dissolve — revisar")
else:
    print("✓ Dissolve por cod_ine correcto: cada municipio es una única geometría")

nucleos_diss = nucleos_diss.merge(padron, left_on="municipio_key",
                                   right_on="municipio", how="left")
nucleos_diss["poblacion"] = nucleos_diss["poblacion"].fillna(0).astype(int)
nucleos_diss["buffer_m"]  = nucleos_diss["poblacion"].apply(buffer_por_poblacion)

print(f"Municipios sin match padrón: {(nucleos_diss['poblacion']==0).sum()}")
print(nucleos_diss[["d_muni_ine", "poblacion", "buffer_m"]].sort_values("poblacion", ascending=False).head(10))

sin_match = nucleos_diss[nucleos_diss["poblacion"] == 0]
if len(sin_match) > 0:
    print(f"\n⚠ {len(sin_match)} municipios sin match:")
    print(sin_match["d_muni_ine"].tolist())

nucleos_diss["geometry"] = nucleos_diss.apply(
    lambda row: row.geometry.buffer(row["buffer_m"]), axis=1
)

BUFFER_GPKG = RAW_DIR / "buffer_nucleos_urbanos.gpkg"
nucleos_diss.to_file(BUFFER_GPKG, driver="GPKG")
print(f"Guardado: {BUFFER_GPKG}")

Descargando padrón INE...
Municipios con población: 202
✓ Dissolve por cod_ine correcto: cada municipio es una única geometría
Municipios sin match padrón: 0
             d_muni_ine  poblacion  buffer_m
91               HUESCA      55454      2000
109              MONZON      18525      2000
37            BARBASTRO      17807      2000
82                FRAGA      15576      2000
95                 JACA      14024      2000
49              BINEFAR      10235      2000
135          SABINANIGO       9695      2000
147            SARINENA       4113      2000
155  TAMARITE DE LITERA       3464      2000
87                GRAUS       3400      2000
Guardado: ..\..\data\raw\06_clasificacion_suelo\buffer_nucleos_urbanos.gpkg


In [4]:
for nombre in ["PERALTA DE ALCOFEA", "SARINENA"]:
    row = nucleos_diss[nucleos_diss["d_muni_ine"] == nombre].iloc[0]
    geom = row.geometry
    print(f"\n{nombre} — {len(geom.geoms)} partes:")
    for i, part in enumerate(geom.geoms):
        print(f"  parte {i}: área = {part.area:,.0f} m²  (radio equivalente ≈ {(part.area/3.1416)**0.5:,.0f} m)")


PERALTA DE ALCOFEA — 3 partes:
  parte 0: área = 4,179,236 m²  (radio equivalente ≈ 1,153 m)
  parte 1: área = 4,429,015 m²  (radio equivalente ≈ 1,187 m)
  parte 2: área = 5,961,418 m²  (radio equivalente ≈ 1,378 m)

SARINENA — 5 partes:
  parte 0: área = 29,347,844 m²  (radio equivalente ≈ 3,056 m)
  parte 1: área = 49,125,592 m²  (radio equivalente ≈ 3,954 m)
  parte 2: área = 15,456,928 m²  (radio equivalente ≈ 2,218 m)
  parte 3: área = 17,732,921 m²  (radio equivalente ≈ 2,376 m)
  parte 4: área = 17,236,403 m²  (radio equivalente ≈ 2,342 m)


In [5]:
# --- Mapa SOLO de buffers de núcleos urbanos (con límite de provincia) ---
DELIM_DIR = Path("../../data/raw/delimitations")
provincia = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:4326")

def geom_to_coords_multi(geom):
    """Devuelve una lista de anillos exteriores (uno por cada polígono del multipolígono)."""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        return [list(p.exterior.coords) for p in geom.geoms]
    return None

nucleos_buffer = nucleos_diss.to_crs("EPSG:4326")
nucleos_buffer["geometry"] = nucleos_buffer["geometry"].simplify(0.0006, preserve_topology=True)
nucleos_buffer["coordinates"] = nucleos_buffer["geometry"].apply(geom_to_coords_multi)
nucleos_buffer = nucleos_buffer[nucleos_buffer["coordinates"].notna()].copy()

# Explode: una fila por cada anillo de polígono (necesario para MultiPolygon en pydeck)
buffer_rows = []
for _, row in nucleos_buffer.iterrows():
    for ring in row["coordinates"]:
        buffer_rows.append({
            "coordinates": [ring],
            "d_muni_ine": row["d_muni_ine"],
            "poblacion": row["poblacion"],
            "buffer_m": row["buffer_m"],
        })

df_buffer = pd.DataFrame(buffer_rows)
print(f"Polígonos de buffer a dibujar: {len(df_buffer)}")

# ─────────────────────────────────────────────────────────
# PALETA — dos tonos fríos, alta legibilidad sobre basemap oscuro,
# relleno con opacidad baja + contorno definido a opacidad plena
# (evita que los buffers cercanos se fundan en manchas)
# ─────────────────────────────────────────────────────────
FILL_1000 = [86,  180, 189, 60]    # teal claro, relleno tenue
LINE_1000 = [110, 210, 220, 235]   # teal, contorno nítido
FILL_2000 = [232, 178, 74,  70]    # ámbar, relleno tenue
LINE_2000 = [244, 196, 108, 235]   # ámbar, contorno nítido

def color_por_buffer(bm, kind):
    if kind == "fill":
        return FILL_1000 if bm == 1000 else FILL_2000
    return LINE_1000 if bm == 1000 else LINE_2000

df_buffer["fill_color"] = df_buffer["buffer_m"].apply(lambda b: color_por_buffer(b, "fill"))
df_buffer["line_color"] = df_buffer["buffer_m"].apply(lambda b: color_por_buffer(b, "line"))

layer_buffer = pdk.Layer(
    "PolygonLayer",
    data=df_buffer,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color="line_color",
    stroked=True,
    filled=True,
    line_width_min_pixels=1.3,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    highlight_color=[255, 255, 255, 60],
)

# --- Límite de la provincia ---
boundary_features = []
for geom in provincia.geometry:
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for p in polys:
        boundary_features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {}
        })

geojson_boundary = {"type": "FeatureCollection", "features": boundary_features}

layer_boundary = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_boundary,
    stroked=True,
    filled=False,
    get_line_color=[235, 235, 240, 210],
    get_line_width=650,
    line_width_min_pixels=1.6,
)

INITIAL_ZOOM = 7.6
INITIAL_LAT = 42.05

view = pdk.ViewState(
    longitude=-0.08,
    latitude=INITIAL_LAT,
    zoom=INITIAL_ZOOM,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <div style="font-family:'Inter','Segoe UI',sans-serif;line-height:1.5">
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Población:</b> {poblacion}<br/>
        <b>Buffer aplicado:</b> {buffer_m} m
        </div>
    """,
    "style": {
        "backgroundColor": "rgba(20,22,26,0.92)",
        "color": "#f2f2f2",
        "fontSize": "12.5px",
        "padding": "8px 10px",
        "borderRadius": "6px",
        "border": "1px solid rgba(255,255,255,0.12)",
    },
}

# ─────────────────────────────────────────────────────────
# Escala aproximada (válida para el zoom/latitud inicial)
# 156543.03392 * cos(lat) / 2^zoom  →  metros por píxel (Web Mercator)
# ─────────────────────────────────────────────────────────
import math
m_per_px = 156543.03392 * math.cos(math.radians(INITIAL_LAT)) / (2 ** INITIAL_ZOOM)
scale_km_options = [5, 10, 20, 25, 50, 100]
target_px = 110
scale_km = min(scale_km_options, key=lambda k: abs((k * 1000 / m_per_px) - target_px))
scale_px = round((scale_km * 1000) / m_per_px)

OVERLAY_HTML = f"""
<style>
  .map-panel {{
    font-family: 'Inter','Segoe UI',sans-serif;
    background: rgba(18,20,24,0.82);
    backdrop-filter: blur(6px);
    color: #f2f2f2;
    border: 1px solid rgba(255,255,255,0.10);
    border-radius: 10px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.35);
  }}
</style>

<div class="map-panel" style="position:fixed;top:18px;left:18px;padding:14px 20px;
            max-width:360px;z-index:9999;">
  <div style="font-size:16px;font-weight:600;letter-spacing:0.2px;">
    Buffers de núcleos urbanos — Huesca
  </div>
  <div style="font-size:12px;color:rgba(255,255,255,0.6);margin-top:3px;">
    Zonas de exclusión para ubicación de planta de biometano
  </div>
</div>

<div class="map-panel" style="position:fixed;bottom:26px;left:18px;padding:14px 18px;
            font-size:13px;z-index:9999;line-height:2;">
  <div style="font-weight:600;font-size:12.5px;color:rgba(255,255,255,0.75);
              text-transform:uppercase;letter-spacing:0.5px;margin-bottom:8px;">
    Leyenda
  </div>
  <div>
    <span style="display:inline-block;width:13px;height:13px;border-radius:3px;
                 margin-right:9px;vertical-align:middle;
                 background:rgba({FILL_1000[0]},{FILL_1000[1]},{FILL_1000[2]},0.55);
                 border:1.5px solid rgb({LINE_1000[0]},{LINE_1000[1]},{LINE_1000[2]});"></span>
    Buffer 1.000 m <span style="color:rgba(255,255,255,0.55);">(&lt; 3.000 hab)</span>
  </div>
  <div>
    <span style="display:inline-block;width:13px;height:13px;border-radius:3px;
                 margin-right:9px;vertical-align:middle;
                 background:rgba({FILL_2000[0]},{FILL_2000[1]},{FILL_2000[2]},0.55);
                 border:1.5px solid rgb({LINE_2000[0]},{LINE_2000[1]},{LINE_2000[2]});"></span>
    Buffer 2.000 m <span style="color:rgba(255,255,255,0.55);">(&ge; 3.000 hab)</span>
  </div>
</div>

<div class="map-panel" style="position:fixed;bottom:26px;right:18px;padding:12px 18px;
            z-index:9999;">
  <div style="display:flex;align-items:center;gap:14px;">
    <svg width="30" height="30" viewBox="0 0 30 30">
      <line x1="15" y1="26" x2="15" y2="6" stroke="#f2f2f2" stroke-width="1.6"/>
      <polygon points="15,3 11,12 19,12" fill="#f2f2f2"/>
      <text x="15" y="0" font-size="9" fill="#f2f2f2" text-anchor="middle"
            font-family="Inter,sans-serif" font-weight="600">N</text>
    </svg>
    <div>
      <div style="width:{scale_px}px;height:0;border-top:2px solid #f2f2f2;position:relative;">
        <div style="position:absolute;left:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
        <div style="position:absolute;right:0;top:-7px;width:1.5px;height:6px;background:#f2f2f2;"></div>
      </div>
      <div style="font-size:11px;color:rgba(255,255,255,0.75);margin-top:4px;">
        ≈ {scale_km} km <span style="color:rgba(255,255,255,0.45);">(zoom inicial)</span>
      </div>
    </div>
  </div>
</div>

<div style="position:fixed;top:18px;right:18px;font-family:'Inter','Segoe UI',sans-serif;
            font-size:10.5px;color:rgba(255,255,255,0.45);z-index:9999;text-align:right;">
  Fuente: INE (población), delimitación provincial Huesca<br/>
  TFM — Análisis de idoneidad, planta de biometano
</div>
"""

deck = pdk.Deck(
    layers=[layer_buffer, layer_boundary],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_buffer_poblacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", OVERLAY_HTML + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Polígonos de buffer a dibujar: 374
Abre: ..\..\docs\map\06_clasificacion_suelo\mapa_buffer_poblacion_huesca.html
